# 真实 API：CoVe 验证链检测器

真实生成基线回答与验证问答，统计与草稿冲突的验证答案比例；默认词面回退，接入 NLI 后按蕴含判定冲突。

## 真实集成前提

安装：`python3 -m pip install -e ".[openai]"`（可选 `.[nli]` 提升冲突判定质量）。

代码单元没有预执行。填写 API 凭据或本地模型绝对路径后，从上到下运行；不要将 API key 保存到 notebook。

## 参数说明

- `max_questions: int = 5`：验证问题数（1-10）。
- `contradiction_threshold: float = 0.5`：NLI 矛盾判定阈值。
- `min_similarity: float = 0.2`：词面回退的冲突判定阈值。
- `TEMPERATURE: float = 0.8`：采样温度。

## 步骤 1：配置真实 API

In [ ]:
from pathlib import Path
import sys
repo_root = Path.cwd()
if not (repo_root / "src" / "hallucination").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

import os
from getpass import getpass
from openai import OpenAI
from hallucination.detection.backends.api import OpenAIResponsesBackend
OPENAI_MODEL = "gpt-4.1-mini"
TEMPERATURE = 0.8
api_key = os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API key: ")
base_url = os.environ.get("OPENAI_BASE_URL")
client = OpenAI(api_key=api_key, **({"base_url": base_url} if base_url else {}))
backend = OpenAIResponsesBackend(OPENAI_MODEL, client=client, temperature=TEMPERATURE)

## 步骤 2：构建 CoVe 检测器

In [ ]:
from hallucination.detection import DetectionExample
from hallucination.detection.blackbox.cove import CoVeScorer
cove = CoVeScorer(backend, max_questions=5)

## 步骤 3：交互运行并读取冲突明细

In [ ]:
prompt = input("Prompt：").strip()
response = input("待检回答（直接回车则由后端生成）：").strip() or backend.generate(prompt)
result = cove.score(DetectionExample(prompt=prompt, response=response))
print("verifier:", result.raw["verifier"], "| conflict_rate:", result.raw["conflict_rate"])
for item in result.raw["comparisons"]:
    print(item["question"], "->", item["verification_answer"], "| conflict:", item["conflict"])

## 结果解读与限制

CoVe 检测完全依赖模型自身知识，是自验证而非外部事实核查；它只报告冲突率，不重写答案。